In [ ]:
import workers   # must come first: it limits each process to one maths thread

import os, time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Settings ---------------------------------------------------------------
DATA_DIR = "."                  # folder holding the parquet files
OUT_DIR = "prepared"            # working files go here
MAX_FILES = 9                   # how many monthly shards to use
SAMPLE_FRACTION = 1             # keep all rows
FEATURES = ["trip_distance"]    # input columns for the model
MAX_WORKERS = 3                 # most worker processes allowed at once
# -----------------------------------------------------------------------------

shard_files = sorted(Path(DATA_DIR).glob("yellow_tripdata_*.parquet"))[:MAX_FILES]
shard_ids = list(range(len(shard_files)))
N_WORKERS = min(MAX_WORKERS, len(shard_files))

print(len(shard_files), "shards:", [f.name for f in shard_files])
print("shard_ids:", shard_ids, "| workers:", N_WORKERS, "| cores on this machine:", os.cpu_count())

In [ ]:
Path(OUT_DIR).mkdir(exist_ok=True)
n = len(shard_files)

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    stats = list(pool.map(workers.prepare_shard, shard_files, shard_ids,
                          [OUT_DIR] * n, [FEATURES] * n, [SAMPLE_FRACTION] * n))
print(f"Prepared in {time.perf_counter() - start:.1f}s")
rows = sum(s["rows_kept"] for s in stats)
print(f"Rows kept: {rows}")